In [1]:
import subprocess
from pathlib import Path

repo_path = Path("./langchain-docs").resolve()

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "--depth", "1",
            "--branch", "main",
            "https://github.com/langchain-ai/docs",
            str(repo_path),
        ],
        check=True,
    )

print("저장소 위치:", repo_path)

저장소 위치: C:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\ch06_advanced_rag\langchain-docs


In [2]:
# 위 셀에서 설정한 repo_path를 사용
if repo_path.name != "langchain-docs" or not repo_path.is_dir():
    raise ValueError(f"정리 대상 폴더를 확인하세요: {repo_path}")

deleted_count = 0

for path in repo_path.rglob("*"):
    if path.is_file() and path.suffix != ".mdx":
        path.chmod(0o666)  # Windows의 읽기 전용 파일도 삭제 가능하게 처리
        path.unlink()
        deleted_count += 1

# 가장 깊은 폴더부터 빈 폴더 삭제
directories = sorted(
    (path for path in repo_path.rglob("*")
     if path.is_dir() and not path.is_symlink()),
    key=lambda path: len(path.parts),
    reverse=True,
)

for directory in directories:
    if not any(directory.iterdir()):
        directory.rmdir()

print(f"삭제한 파일: {deleted_count}개")
print(f"남은 MDX 파일: {sum(1 for _ in repo_path.rglob('*.mdx'))}개")

삭제한 파일: 0개
남은 MDX 파일: 2532개


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="./langchain-docs",
    glob="**/*.mdx",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    use_multithreading=True,
)
documents = loader.load()
print(len(documents))

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_36856\2494807805.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


2532


In [7]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")

text_splitter=RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"원본 문서: {len(documents)}개")
print(f"분할된 청크: {len(chunks)}개")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4739 > 512). Running this sequence through the model will result in indexing errors


원본 문서: 2532개
분할된 청크: 18413개


In [8]:
input_lengths = [
    len(
        tokenizer.encode(
            "passage: " + chunk.page_content.replace("\n", " "),
            add_special_tokens=True,
            truncation=False,  # 잘라내지 않고 실제 길이 확인
        )
    )
    for chunk in chunks
]

max_length = max(input_lengths, default=0)

print(f"가장 긴 모델 입력: {max_length}토큰")
assert max_length <= 512, "512토큰을 넘는 청크가 있습니다."

가장 긴 모델 입력: 404토큰


In [9]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "prompt": "passage: ",
        "normalize_embeddings": True,
    },
    query_encode_kwargs={
        "prompt": "query: ",
        "normalize_embeddings": True,
    }
)
db = Chroma.from_documents(chunks,embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1581.31it/s]


In [10]:
# 단순한 RAG를 구현해 보겠다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.

문맥: """
{context}
"""

질문: {question}
''')

model = ChatOpenAI(model="gpt-6-luna")
retriever = db.as_retriever()

chain = {
    "question": RunnablePassthrough(),
    "context": retriever,
} | prompt | model | StrOutputParser()

chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 LLM 기반 애플리케이션을 만들기 위한 프레임워크입니다. 모델과 도구, 외부 데이터 등을 연결해 단순한 텍스트 생성을 넘어 복잡한 작업 흐름과 에이전트를 구성할 수 있습니다.\n\n주요 특징은 다음과 같습니다.\n\n- 여러 모델 제공자의 입력·출력을 표준화해 모델을 쉽게 바꿀 수 있습니다.\n- 모델, 도구, 데이터 처리 등을 위한 구성 요소와 통합 기능을 제공합니다.\n- 에이전트를 직접 구성할 수 있는 유연한 추상화를 제공하며, 에이전트는 LangGraph 위에서 동작합니다.\n- 더 세밀한 흐름 제어, 지속 실행, 스트리밍, 사람의 검토가 필요한 경우에는 저수준 오케스트레이션 프레임워크인 LangGraph를 함께 사용할 수 있습니다.\n- LangSmith는 에이전트의 실행을 추적하고 디버깅·평가하는 데 활용할 수 있습니다.'